# Experiment: SpectralBio Final Accept Hardening

This notebook is the strongest single supplementary rerun designed to close the remaining SpectralBio reviewer risks as far as the evidence allows.

## What it does
- Scans ClinVar globally to rank genes by support under pre-declared rules.
- Builds a support-ranked multigene panel instead of a hand-picked panel.
- Re-runs the TP53 nested-CV leakage audit.
- Re-runs the TP53 stronger-backbone comparison.
- Runs support-first nested CV on the strongest non-anchor genes.
- Recommends a second canonical benchmark candidate under explicit rules.
- Runs a deep checkpoint sweep on TP53 plus the strongest follow-up genes, including ESM2-3B when enabled.
- Writes a final machine-readable bundle and downloads it.

## Honest goal
This notebook is meant to maximize the chance that the remaining weaknesses become either resolved or clearly bounded. It cannot guarantee a positive result in advance; if a larger model or broader panel weakens the thesis, we keep that result and use it honestly.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioFinalAcceptHardening')

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs will stay in the Colab VM unless you override OUTPUT_ROOT later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
REPO_ROOT = Path('/content/Stanford-Claw4s')
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_v3_complete'

if not REPO_ROOT.exists():
    print('Cloning repo...')
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    print('Installing notebook dependencies without replacing Colab torch...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        'numpy==2.1.3',
        'plotly==5.24.1',
        'pyyaml==6.0.2',
        'scikit-learn==1.5.2',
        'scipy==1.14.1',
        'transformers==4.49.0',
        'pandas',
        'matplotlib',
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Restarting runtime once to reload packages cleanly...')
    os.kill(os.getpid(), 9)
else:
    print('Bootstrap marker found; skipping reinstall.')

try:
    from importlib import metadata as importlib_metadata
except ImportError:
    import importlib_metadata

def _package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return 'not installed'

print('Repo ready at', REPO_ROOT)
print('Repo branch:', REPO_BRANCH)
print('torch version:', _package_version('torch'))
print('torchvision version:', _package_version('torchvision'))
print('transformers version:', _package_version('transformers'))


## Plan

- Primary target: reduce the remaining scope, leakage, model-size, and benchmark-fragility criticisms as much as one notebook can.
- Selection discipline: support-first, not score-first.
- Large-model discipline: use 650M and 3B only on the deepest validation genes so the T4 is spent where it matters most.
- Success condition: produce a final bundle that either upgrades the claim materially or shows exactly which limits remain.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        recommend_second_benchmark_candidate,
        run_accept_hardening_suite,
    )
    from spectralbio.supplementary.reject_recovery import (
        create_reject_recovery_zip,
        write_experiment_log,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        recommend_second_benchmark_candidate,
        run_accept_hardening_suite,
    )
    from spectralbio.supplementary.reject_recovery import (
        create_reject_recovery_zip,
        write_experiment_log,
    )

USE_3B = True
MIN_TOTAL = 60
MIN_PER_CLASS = 20
MAX_ADDITIONAL_GENES = 8
SHORTLIST_NON_ANCHOR_GENES = 3
LARGE_MODEL_GENE_LIMIT = 4
CHECKPOINT_EVERY = 10
BOOTSTRAP_REPLICATES = 1000
OVERWRITE = False

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'final_accept_hardening'
)

model_names = ('facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D') if USE_3B else ('facebook/esm2_t33_650M_UR50D',)

config = AcceptHardeningConfig(
    stronger_model_names=model_names,
    min_total=MIN_TOTAL,
    min_per_class=MIN_PER_CLASS,
    max_additional_genes=MAX_ADDITIONAL_GENES,
    shortlist_non_anchor_genes=SHORTLIST_NON_ANCHOR_GENES,
    large_model_gene_limit=LARGE_MODEL_GENE_LIMIT,
    checkpoint_every=CHECKPOINT_EVERY,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
    overwrite=OVERWRITE,
)

paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
print(paths)
print(config)


In [ ]:
summary = run_accept_hardening_suite(paths, config)
summary_path = paths.root / 'final_accept_hardening_summary.json'
print('Final summary written to:', summary_path)
print('Recommended second benchmark candidate:', summary['second_benchmark_candidate']['recommended_gene'])
print('Deep checkpoint genes:', summary['checkpoint_sweep']['genes'])


In [ ]:
support_table = pd.read_csv(paths.root / 'support_scan' / 'clinvar_gene_support_table.csv')
support_ranked = pd.read_csv(paths.multigene / 'support_ranked_gene_table.csv')
multigene_summary = pd.read_csv(paths.multigene / 'multigene_summary.csv')
checkpoint_long = pd.read_csv(paths.root / 'checkpoint_sweep' / 'checkpoint_sweep_long.csv')

display(support_table.head(20))
display(support_ranked)
display(multigene_summary.sort_values(['delta_pair_vs_ll', 'n_total'], ascending=[False, False]))
display(checkpoint_long.sort_values(['gene', 'model_rank']))


In [ ]:
candidate = summary['second_benchmark_candidate']
pd.DataFrame(candidate['candidates'])


In [ ]:
notes = [
    'This suite uses support-ranked panel construction instead of a hand-picked candidate list.',
    'If 3B weakens the thesis, keep the result and update the manuscript honestly.',
    f"Recommended second benchmark candidate: {summary['second_benchmark_candidate']['recommended_gene']}",
]
log_path = write_experiment_log(
    paths,
    completed_experiments=[
        'Global ClinVar support scan',
        'Support-ranked panel build',
        'TP53 nested CV',
        'TP53 backbone comparison',
        'Support-ranked multigene panel',
        'Shortlist nested CV',
        'Second benchmark candidate recommendation',
        'Deep checkpoint sweep',
    ],
    skipped_experiments=[],
    notes=notes,
)
zip_path = create_reject_recovery_zip(paths, bundle_name='spectralbio_final_accept_hardening_bundle')
print('Experiment log:', log_path)
print('ZIP ready at:', zip_path)


In [ ]:
print(f'ZIP ready at: {zip_path}')

if zip_path.exists():
    from google.colab import files
    files.download(str(zip_path))
else:
    print('ZIP not found:', zip_path)
